# Limpieza del dataset — Predicción de `Unit Price (USD)`

**Parte 1: dataset listo para entrenamiento**

Dataset: `line_items_export(7).csv` (line items de manufactura).  
**Target final: `log_unit_price`** (= `log(Total Price / Quantity)`).

Decisión de target: `Total Price` está dominado por `Quantity` (corr ≈ 0.96), lo que enmascara el efecto de las features de complejidad/material. Predecir `Unit Price` aísla el precio por pieza; `Quantity` se mantiene como feature para que el modelo aprenda los descuentos por volumen. Aplicamos `log` para que los descuentos multiplicativos se vuelvan aditivos.

Pasos:
- Cargar el CSV (limpiar padding de espacios en headers/valores).
- Overview rápido.
- Filtrar filas sin `Total Price` válido y derivar `Unit Price`.
- Eliminar columnas:
  - **Grupo A** — identificadores / metadatos.
  - **Grupo B** — leakage directo del target (`Total Price`, `Total Partner Cost`).
  - **Grupo C** — leakage indirecto en USD (componentes de costo del pricing engine).
  - **Grupo D** — baja variabilidad y multicolinealidad.
  - **Grupo E** — leakage indirecto en horas + tarifa + multiplicadores + binización categórica redundante.
- Eliminar **outliers en el target** (IQR sobre log).
- **Transformación log del target**.
- **Tratamiento de categóricas** (Tolerance → numérico, agrupar raras, convertir a `category`).
- **Eliminar duplicados**.
- **Feature engineering**: `log(Quantity)`.
- **Inventario final de missingness**.
- Guardar parquet intermedio + CSV de preview.

**Output final**: dataset con specs categóricas (como `category` dtype), geometría del CAD, `Quantity` + `log_quantity`, target log, y target original — listo para entrenar con LightGBM / XGBoost / CatBoost.

In [106]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

DATA_PATH = Path('line_items_export(7).csv')
OUT_PATH  = Path('line_items_step1.parquet')

## 1. Carga

El export tiene padding de espacios en headers y en valores string. Lo limpiamos al cargar.

In [107]:
df = pd.read_csv(DATA_PATH, skipinitialspace=True)

df.columns = df.columns.str.strip()
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].str.strip().replace({'': np.nan})

print(f'Shape: {df.shape}')
df.head(3)

Shape: (3238, 65)


,Radii ID,Title,Company,Line Item Name,Material,Technology,Tolerance,Post Production,Finishing,Heat Treatment,Quantity,Unit Price (USD),Total Price (USD),Total Partner Cost (USD),Status,Created,Blueprint Format,part_volume_cm3,weight_g,stock_volume_cm3,removed_volume_cm3,material_removal_pct,stock_dimensions_mm.x,stock_dimensions_mm.y,stock_dimensions_mm.z,faces_qty,aspect_ratio,thin_ratio,plane_ratio,cyl_ratio,complex_ratio,total_holes,machining_hours,programming_hours,num_setups,setup_hours,setup_time_per_setup,roughing_hours,finishing_hours,material_dependent_hours,effective_hourly_rate (USD),complexity_factor,mesh_complexity_penalty,complexity_risk_mult,total_risk_mult,tolerance_tier,tolerance_tier_mult,tolerance_risk_mult,axis_requirement,edm_required,part_type,material_cost (USD),machining_cost (USD),feature_cost (USD),finishing_cost (USD),heat_treatment_cost (USD),post_production_cost (USD),overhead (USD),total_cost_floor (USD),material_nesting_factor,machining_batch_factor,overhead_batch_factor,feature_machining_hours,tool_change_hours,amortized_fixed_hours
0,RAD-6281,Manufacturing Request - 856A3790G01_B,Radii,856A3790G01_B,6061 Aluminum,CNC Multiaxis,0.25,NaN,As Machined,NaN,1,NaN,NaN,NaN,Created,2026-05-06 00:02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,RAD-6280,Grippers_2np,Radii,Grippper,1045 Steel,CNC Multiaxis,0.127,Other,As Machined,NaN,1,325.0,325.0,NaN,Created,2026-05-05 17:38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,RAD-6280,Grippers_2np,Radii,Gripper,1045 Steel,CNC Multiaxis,0.127,Other,As Machined,NaN,1,325.0,325.0,NaN,Created,2026-05-05 17:38,.png,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Overview rápido

In [108]:
overview = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(1),
    'n_unique': df.nunique(dropna=True),
    'sample': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
overview.sort_values('pct_missing', ascending=False)

,dtype,n_missing,pct_missing,n_unique,sample
Heat Treatment,object,2818,87.0,6,Tempering
Total Partner Cost (USD),float64,2798,86.4,321,293.2806
part_type,object,2730,84.3,4,rotational
axis_requirement,object,2724,84.1,1,3-axis
complex_ratio,float64,2724,84.1,128,0.0
...,...,...,...,...,...
Quantity,int64,0,0.0,153,1
Status,object,0,0.0,19,Created
Created,object,0,0.0,632,2026-05-06 00:02
Title,object,0,0.0,945,Manufacturing Request - 856A3790G01_B


## 3. Filtrar filas y derivar el target

Filtramos filas con `Total Price` y `Quantity` válidos, y derivamos `Unit Price = Total Price / Quantity` como nuevo target. Sobreescribir la columna existente es intencional: la versión del CSV tiene inconsistencias en algunas filas; nuestra versión es siempre consistente con `Total Price`.

In [109]:
TARGET = 'Unit Price (USD)'

n_before = len(df)
df = df[df['Total Price (USD)'].notna() & (df['Total Price (USD)'] > 0) & (df['Quantity'] > 0)].copy()

# Derivar Unit Price = Total Price / Quantity (overwrite del Unit Price del CSV)
df[TARGET] = df['Total Price (USD)'] / df['Quantity']

n_after = len(df)
print(f'Filas válidas: {n_after} / {n_before}  (drop {n_before - n_after})')
df[TARGET].describe()

Filas válidas: 2110 / 3238  (drop 1128)


count    2.110000e+03
mean     5.001030e+03
std      1.070082e+05
min      1.949608e-02
25%      6.131312e+01
50%      1.590981e+02
75%      3.700000e+02
max      3.123021e+06
Name: Unit Price (USD), dtype: float64

## 4. Corte de columnas

### Grupo A — Identificadores / metadatos
Sin señal predictiva para el precio:
- `Radii ID`, `Title`, `Line Item Name` — texto libre / ID único
- `Company` — *nota: NO es constante (varias empresas). La quitamos en v0 por ser metadata de quién emitió el pedido, pero podría reincorporarse como feature categórica si vemos que el precio varía sistemáticamente por cliente.*
- `Status` — estado operativo del registro
- `Created` — timestamp (lo descartamos en v0; reconsiderar si quisiéramos modelar tendencias temporales)
- `Blueprint Format` — formato del archivo, irrelevante para el precio

### Grupo B — Leakage directo del nuevo target (`Unit Price`)
Como `Unit Price = Total Price / Quantity`, conservar `Total Price` filtra el target:
- `Total Price (USD)` — define el target.
- `Total Partner Cost (USD)` — costo al partner, altísima correlación con cualquier precio.

### Grupo C — Leakage indirecto (componentes de costo del pricing engine)
El pricing engine calcula el precio final sumando estos componentes y aplicando markup. Si los dejamos, el modelo aprende la fórmula del motor; si los quitamos, aprende a cotizar desde **specs + geometría**, que es lo útil para autocotización en frío. Eliminamos:
- `material_cost (USD)`, `machining_cost (USD)`, `feature_cost (USD)`, `finishing_cost (USD)`
- `heat_treatment_cost (USD)`, `post_production_cost (USD)`, `overhead (USD)`
- `total_cost_floor (USD)` (suma de todos los anteriores; el target es función de éste)


In [110]:
# Validación: ¿estas categóricas aportan? (justifica eliminarlas en Grupo A)
for c in ['Company', 'Status', 'Blueprint Format']:
    print(f'{c}: {df[c].value_counts(dropna=False).head().to_dict()}')

Company: {'Radii': 675, 'Tecnomaquinados': 313, 'Brose': 295, 'Schaeffler Mexico': 177, 'Mankiewickz': 126}
Status: {'Offer Sent': 642, 'Completed': 472, 'Created': 343, 'Needs Review': 200, 'Expired': 162}
Blueprint Format: {'.stp': 747, '.step': 700, nan: 440, '.png': 78, '.pdf': 77}


In [111]:
drop_group_a = [
    'Radii ID',
    'Title',
    'Line Item Name',
    'Company',
    'Status',
    'Created',
    'Blueprint Format',
]

drop_group_b = [
    'Total Price (USD)',         # leakage: target = Total Price / Quantity
    'Total Partner Cost (USD)',
]

drop_group_c = [
    'material_cost (USD)',
    'machining_cost (USD)',
    'feature_cost (USD)',
    'finishing_cost (USD)',
    'heat_treatment_cost (USD)',
    'post_production_cost (USD)',
    'overhead (USD)',
    'total_cost_floor (USD)',
]

to_drop = [c for c in drop_group_a + drop_group_b + drop_group_c if c in df.columns]
df_step1 = df.drop(columns=to_drop)

print(f'Columnas eliminadas ({len(to_drop)}):')
print(f'  Grupo A ({len(drop_group_a)}): {drop_group_a}')
print(f'  Grupo B ({len(drop_group_b)}): {drop_group_b}')
print(f'  Grupo C ({len(drop_group_c)}): {drop_group_c}')
print(f'\nShape antes: {df.shape}  ->  después: {df_step1.shape}')
df_step1.head(3)

Columnas eliminadas (17):
  Grupo A (7): ['Radii ID', 'Title', 'Line Item Name', 'Company', 'Status', 'Created', 'Blueprint Format']
  Grupo B (2): ['Total Price (USD)', 'Total Partner Cost (USD)']
  Grupo C (8): ['material_cost (USD)', 'machining_cost (USD)', 'feature_cost (USD)', 'finishing_cost (USD)', 'heat_treatment_cost (USD)', 'post_production_cost (USD)', 'overhead (USD)', 'total_cost_floor (USD)']

Shape antes: (2110, 65)  ->  después: (2110, 48)


,Material,Technology,Tolerance,Post Production,Finishing,Heat Treatment,Quantity,Unit Price (USD),part_volume_cm3,weight_g,stock_volume_cm3,removed_volume_cm3,material_removal_pct,stock_dimensions_mm.x,stock_dimensions_mm.y,stock_dimensions_mm.z,faces_qty,aspect_ratio,thin_ratio,plane_ratio,cyl_ratio,complex_ratio,total_holes,machining_hours,programming_hours,num_setups,setup_hours,setup_time_per_setup,roughing_hours,finishing_hours,material_dependent_hours,effective_hourly_rate (USD),complexity_factor,mesh_complexity_penalty,complexity_risk_mult,total_risk_mult,tolerance_tier,tolerance_tier_mult,tolerance_risk_mult,axis_requirement,edm_required,part_type,material_nesting_factor,machining_batch_factor,overhead_batch_factor,feature_machining_hours,tool_change_hours,amortized_fixed_hours
1,1045 Steel,CNC Multiaxis,0.127,Other,As Machined,NaN,1,325.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1045 Steel,CNC Multiaxis,0.127,Other,As Machined,NaN,1,325.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6061-T6 Aluminum,CNC Multiaxis,0.025,Electroless Nickel Plating,As Machined,NaN,20,150.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [112]:
# Eliminar registros sin part_volume_cm3 (sin análisis CAD)
n_before = len(df_step1)
df_step1 = df_step1[df_step1['part_volume_cm3'].notna()].copy()
n_after = len(df_step1)

print(f'Registros antes:    {n_before}')
print(f'Registros después:  {n_after}')
print(f'Eliminados:         {n_before - n_after}')

Registros antes:    2110
Registros después:  429
Eliminados:         1681


## 5. Identificar columnas problemáticas

Después de filtrar a registros con CAD analizado, detectamos columnas problemáticas en dos ejes:

- **Baja variación / alto missing**: si una columna es casi constante (un valor domina >95%) o sigue >90% nula, no aporta señal.
- **Alta correlación entre features**: pares numéricos con `|r| > 0.9` indican multicolinealidad (una de las dos es redundante).

En este paso **sólo identificamos** — no eliminamos automáticamente. Después de ver la salida decidimos cuáles cortar.

In [113]:
# Baja variación: constantes, casi constantes (un valor domina), y alto missing
def dominant_pct(series):
    s = series.dropna()
    if len(s) == 0:
        return 1.0
    return s.value_counts(normalize=True).iloc[0]

def dominant_value(series):
    s = series.dropna()
    return s.value_counts().index[0] if len(s) > 0 else None

variation = pd.DataFrame({
    'dtype': df_step1.dtypes.astype(str),
    'n_unique': df_step1.nunique(dropna=True),
    'pct_missing': (df_step1.isna().mean() * 100).round(1),
    'dominant_value': df_step1.apply(dominant_value),
    'dominant_pct': (df_step1.apply(dominant_pct) * 100).round(1),
})

variation['flag'] = ''
variation.loc[variation['n_unique'] <= 1, 'flag'] = 'CONSTANT'
variation.loc[variation['pct_missing'] > 90, 'flag'] = 'HIGH_MISSING'
variation.loc[(variation['dominant_pct'] > 95) & (variation['n_unique'] > 1) & (variation['flag'] == ''), 'flag'] = 'NEAR_CONSTANT'

flagged = variation[variation['flag'] != ''].sort_values(['flag', 'dominant_pct'], ascending=[True, False])
print(f'Columnas con baja variación / alto missing: {len(flagged)}')
flagged

Columnas con baja variación / alto missing: 2


,dtype,n_unique,pct_missing,dominant_value,dominant_pct,flag
axis_requirement,object,1,0.0,3-axis,100.0,CONSTANT
edm_required,object,2,0.0,false,99.1,NEAR_CONSTANT


In [114]:
# Multicolinealidad: pares de features numéricas altamente correlacionadas
numeric_df = df_step1.select_dtypes(include=[np.number])
features_only = numeric_df.drop(columns=[TARGET], errors='ignore')

corr = features_only.corr().abs()
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
upper = corr.where(mask)

CORR_THRESHOLD = 0.9
high_corr_pairs = (upper.stack()
                        .reset_index()
                        .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'abs_corr'}))
high_corr_pairs = (high_corr_pairs[high_corr_pairs['abs_corr'] > CORR_THRESHOLD]
                       .sort_values('abs_corr', ascending=False)
                       .reset_index(drop=True))

print(f'Pares con |corr| > {CORR_THRESHOLD}: {len(high_corr_pairs)}')
print()
print('--- Correlación absoluta de cada feature con el target ---')
target_corr = numeric_df.corr()[TARGET].drop(TARGET).abs().sort_values(ascending=False)
print(target_corr.round(3).to_string())
print()
print('--- Pares altamente correlacionados (candidatos a redundancia) ---')
print('Sugerencia: en cada par, conservar la que tenga mayor correlación con el target.')
high_corr_pairs

Pares con |corr| > 0.9: 9

--- Correlación absoluta de cada feature con el target ---
material_dependent_hours       0.830
roughing_hours                 0.825
finishing_hours                0.825
stock_volume_cm3               0.509
removed_volume_cm3             0.475
stock_dimensions_mm.y          0.438
stock_dimensions_mm.x          0.374
stock_dimensions_mm.z          0.355
machining_hours                0.308
mesh_complexity_penalty        0.248
amortized_fixed_hours          0.199
programming_hours              0.143
total_holes                    0.119
tool_change_hours              0.117
total_risk_mult                0.105
feature_machining_hours        0.095
complexity_risk_mult           0.091
setup_hours                    0.085
num_setups                     0.084
overhead_batch_factor          0.082
part_volume_cm3                0.080
weight_g                       0.080
machining_batch_factor         0.073
plane_ratio                    0.063
material_nesting_factor   

,feature_1,feature_2,abs_corr
0,part_volume_cm3,weight_g,0.999997
1,stock_volume_cm3,removed_volume_cm3,0.998541
2,roughing_hours,material_dependent_hours,0.997992
3,tolerance_tier_mult,tolerance_risk_mult,0.997135
4,finishing_hours,material_dependent_hours,0.981892
5,roughing_hours,finishing_hours,0.969712
6,num_setups,setup_hours,0.957667
7,total_holes,complexity_risk_mult,0.946607
8,machining_batch_factor,overhead_batch_factor,0.912764


## 6. Aplicar drops del Grupo D

Decisión basada en el análisis previo:

**Baja variabilidad** (no aportan información):
- `axis_requirement` (constante: 100% `'3-axis'`)
- `edm_required` (casi constante: 99.1% `'false'`)

**Multicolinealidad** (en cada par con `|r| > 0.9`, conservamos uno):

| Par redundante | Conservada | Eliminada |
|---|---|---|
| `part_volume_cm3` ↔ `weight_g` | `part_volume_cm3` | `weight_g` |
| `stock_volume_cm3` ↔ `removed_volume_cm3` | `stock_volume_cm3` | `removed_volume_cm3` |
| `roughing_hours` ↔ `material_dependent_hours` | `material_dependent_hours` | `roughing_hours` |
| `tolerance_tier_mult` ↔ `tolerance_risk_mult` | `tolerance_risk_mult` | `tolerance_tier_mult` |
| `finishing_hours` ↔ `material_dependent_hours` | `material_dependent_hours` | `finishing_hours` |
| `num_setups` ↔ `setup_hours` | `setup_hours` | `num_setups` |
| `total_holes` ↔ `complexity_risk_mult` | `total_holes` | `complexity_risk_mult` |
| `machining_batch_factor` ↔ `overhead_batch_factor` | `overhead_batch_factor` | `machining_batch_factor` |

In [115]:
drop_low_variability = [
    'axis_requirement',   # CONSTANT  (100% '3-axis')
    'edm_required',       # NEAR_CONSTANT (99.1% 'false')
]

# Multicolinealidad: en cada par |r| > 0.9, conservamos la feature más correlacionada
# con el target (o más fundamental cuando empatan).
drop_multicollinear = [
    'weight_g',                  # ↔ part_volume_cm3 (r≈1.00)
    'removed_volume_cm3',        # ↔ stock_volume_cm3 (r≈1.00)
    'roughing_hours',            # ↔ material_dependent_hours (r≈1.00)
    'tolerance_tier_mult',       # ↔ tolerance_risk_mult (r≈1.00)
    'finishing_hours',           # ↔ material_dependent_hours (r≈0.98)
    'num_setups',                # ↔ setup_hours (r≈0.96)
    'complexity_risk_mult',      # ↔ total_holes (r≈0.95)
    'machining_batch_factor',    # ↔ overhead_batch_factor (r≈0.91)
]

drop_d = [c for c in drop_low_variability + drop_multicollinear if c in df_step1.columns]
df_step1 = df_step1.drop(columns=drop_d)

print(f'Columnas eliminadas ({len(drop_d)}):')
print(f'  Baja variabilidad ({len(drop_low_variability)}): {drop_low_variability}')
print(f'  Multicolinealidad ({len(drop_multicollinear)}): {drop_multicollinear}')
print(f'\nShape final: {df_step1.shape}')

Columnas eliminadas (10):
  Baja variabilidad (2): ['axis_requirement', 'edm_required']
  Multicolinealidad (8): ['weight_g', 'removed_volume_cm3', 'roughing_hours', 'tolerance_tier_mult', 'finishing_hours', 'num_setups', 'complexity_risk_mult', 'machining_batch_factor']

Shape final: (429, 38)


## 7. Aplicar drops del Grupo E — leakage del pricing engine

Tras revisar las correlaciones contra el nuevo target, las features más correlacionadas son **horas**: `material_dependent_hours: 0.83`, `roughing_hours: 0.82`, `finishing_hours: 0.82`. Esa correlación no es señal predictiva: las horas son insumo del cálculo del pricing engine (`costo ≈ horas × tarifa × multiplicadores`), o sea es la misma fuga que el Grupo C (componentes de costo en USD), pero expresada en horas + multiplicadores.

Eliminamos:

**Horas** (insumos directos del cálculo de costo):
- `machining_hours`, `programming_hours`, `setup_hours`, `setup_time_per_setup`
- `material_dependent_hours`, `feature_machining_hours`, `tool_change_hours`, `amortized_fixed_hours`

**Tarifa**:
- `effective_hourly_rate (USD)` (lo que multiplica las horas para obtener costo)

**Multiplicadores del motor de precios**:
- `complexity_factor`, `mesh_complexity_penalty`
- `total_risk_mult`, `tolerance_risk_mult`
- `material_nesting_factor`, `overhead_batch_factor`

**Bonus — categórica redundante**:
- `tolerance_tier` (binización de `Tolerance` numérico; el modelo de árbol/GBM puede aprender los cortes desde el valor crudo).

Tras este corte, el dataset queda únicamente con inputs *genuinos* del proceso de cotización: specs categóricas, geometría del CAD, y `Quantity`.

In [116]:
drop_engine_hours = [
    'machining_hours',
    'programming_hours',
    'setup_hours',
    'setup_time_per_setup',
    'material_dependent_hours',
    'feature_machining_hours',
    'tool_change_hours',
    'amortized_fixed_hours',
]

drop_engine_rate = [
    'effective_hourly_rate (USD)',
]

drop_engine_multipliers = [
    'complexity_factor',
    'mesh_complexity_penalty',
    'total_risk_mult',
    'tolerance_risk_mult',
    'material_nesting_factor',
    'overhead_batch_factor',
]

drop_redundant_categorical = [
    'tolerance_tier',  # binización redundante de Tolerance numérico
]

drop_e = [c for c in (drop_engine_hours + drop_engine_rate
                      + drop_engine_multipliers + drop_redundant_categorical)
          if c in df_step1.columns]
df_step1 = df_step1.drop(columns=drop_e)

print(f'Columnas eliminadas ({len(drop_e)}):')
print(f'  Horas ({len(drop_engine_hours)}): {drop_engine_hours}')
print(f'  Tarifa ({len(drop_engine_rate)}): {drop_engine_rate}')
print(f'  Multiplicadores ({len(drop_engine_multipliers)}): {drop_engine_multipliers}')
print(f'  Categórica redundante ({len(drop_redundant_categorical)}): {drop_redundant_categorical}')
print(f'\nShape final: {df_step1.shape}')

Columnas eliminadas (16):
  Horas (8): ['machining_hours', 'programming_hours', 'setup_hours', 'setup_time_per_setup', 'material_dependent_hours', 'feature_machining_hours', 'tool_change_hours', 'amortized_fixed_hours']
  Tarifa (1): ['effective_hourly_rate (USD)']
  Multiplicadores (6): ['complexity_factor', 'mesh_complexity_penalty', 'total_risk_mult', 'tolerance_risk_mult', 'material_nesting_factor', 'overhead_batch_factor']
  Categórica redundante (1): ['tolerance_tier']

Shape final: (429, 22)


## 8. Detectar y eliminar outliers en el target

`Unit Price` está muy sesgado (rango $0.02 → $3.1M, mediana $159). Los extremos son casi seguro errores de captura ($0.02/pieza no tiene sentido) o casos rarísimos que dominarían la pérdida del modelo.

**Método**: IQR sobre `log10(Unit Price)`. Usamos log porque la distribución es muy sesgada y log la simetriza, lo cual hace que IQR sea apropiado. Banda estándar `[Q1 − k·IQR, Q3 + k·IQR]` con `k=1.5`.

Estrategia en dos pasos: primero **identificamos** los outliers y los listamos para validar visualmente que son razonables; luego **eliminamos**.

In [117]:
# Detectar outliers en el target con IQR sobre log10(Unit Price)
# Subir IQR_K (p.ej. 3.0) para ser más conservador y mantener más filas.
IQR_K = 1.5

target_log = np.log10(df_step1[TARGET])
q1, q3 = target_log.quantile([0.25, 0.75])
iqr = q3 - q1
LOWER_FENCE = 10 ** (q1 - IQR_K * iqr)
UPPER_FENCE = 10 ** (q3 + IQR_K * iqr)

outlier_mask = (df_step1[TARGET] < LOWER_FENCE) | (df_step1[TARGET] > UPPER_FENCE)

print(f'Fences (USD):        [${LOWER_FENCE:,.2f}, ${UPPER_FENCE:,.2f}]')
print(f'Outliers detectados: {outlier_mask.sum()} / {len(df_step1)} ({outlier_mask.mean()*100:.1f}%)')
print()
print('--- Sample de outliers (los más bajos y los más altos) ---')
preview_cols = [c for c in [TARGET, 'Quantity', 'Material', 'Technology'] if c in df_step1.columns]
sorted_outliers = df_step1.loc[outlier_mask, preview_cols].sort_values(TARGET)
if len(sorted_outliers) <= 20:
    print(sorted_outliers.to_string())
else:
    print(sorted_outliers.head(10).to_string())
    print('  ...')
    print(sorted_outliers.tail(10).to_string())

Fences (USD):        [$1.24, $15,465.92]
Outliers detectados: 12 / 429 (2.8%)

--- Sample de outliers (los más bajos y los más altos) ---
     Unit Price (USD)  Quantity             Material     Technology
649          0.080508      2000           1018 Steel  CNC Multiaxis
891          0.080508      2000                Other  CNC Multiaxis
646          0.086259      4000                Other  CNC Multiaxis
647          0.092010      4000                Other  CNC Multiaxis
449          0.172518        50                Other  CNC Multiaxis
648          0.224273      4000                Other  CNC Multiaxis
890          0.287530      5000                Other  CNC Multiaxis
889          0.339285      2000     6061-T6 Aluminum  CNC Multiaxis
450          0.747578        50                Other  CNC Multiaxis
448          1.035108        50                Other  CNC Multiaxis
692          1.150120      5000  304 Stainless Steel    Sheet Metal
662      37953.960000         1               

In [118]:
# Eliminar outliers usando los fences calculados arriba (LOWER_FENCE / UPPER_FENCE)
n_before = len(df_step1)
df_step1 = df_step1[
    (df_step1[TARGET] >= LOWER_FENCE) & (df_step1[TARGET] <= UPPER_FENCE)
].copy()
n_after = len(df_step1)

print(f'Registros antes:     {n_before}')
print(f'Registros después:   {n_after}')
print(f'Outliers eliminados: {n_before - n_after}')
print()
print('--- Distribución final del target ---')
print(df_step1[TARGET].describe())

Registros antes:     429
Registros después:   417
Outliers eliminados: 12

--- Distribución final del target ---
count      417.000000
mean       525.239249
std       1274.354894
min          1.400000
25%         46.000000
50%        174.000000
75%        450.000000
max      11916.150000
Name: Unit Price (USD), dtype: float64


## 9. Transformación log del target

Aplicamos `log` natural al target porque:
- Los descuentos por volumen son típicamente multiplicativos (-10%, -25%): en log space se vuelven aditivos y más fáciles de aprender.
- Aun después de eliminar outliers, el rango es amplio ($1.4 → $11.9K). Log lo simetriza y reduce la influencia de valores grandes.
- MAE sobre `log` corresponde aproximadamente a MAPE en USD — métrica natural para precios.

Mantenemos también la columna original `Unit Price (USD)` para poder hacer inverse transform al servir el modelo (`np.exp(pred)`).

In [119]:
TARGET_USD = 'Unit Price (USD)'
TARGET = 'log_unit_price'

df_step1[TARGET] = np.log(df_step1[TARGET_USD])

print(f'Nuevo target: {TARGET}')
print(f'(Inverse transform: np.exp({TARGET!r}) → {TARGET_USD!r})')
print()
print(f'--- {TARGET} (log natural) ---')
print(df_step1[TARGET].describe())

Nuevo target: log_unit_price
(Inverse transform: np.exp('log_unit_price') → 'Unit Price (USD)')

--- log_unit_price (log natural) ---
count    417.000000
mean       4.994224
std        1.698616
min        0.336472
25%        3.828641
50%        5.159055
75%        6.109248
max        9.385650
Name: log_unit_price, dtype: float64


## 10. Tratamiento de categóricas

Estrategia:
1. **Convertir `Tolerance` a numérico**: contiene valores tipo `'0.127'`, `'0.025'` que están como string. La pasamos a `float`; los no numéricos (si hay) quedan en `NaN` y se imputan luego.
2. **Agrupar categorías raras**: en cada columna categórica, los valores con ≤5 ocurrencias se agrupan en `'Other_rare'`. Con 417 filas, eso es <1.2% del dataset y evita que el modelo memorice categorías sin señal.
3. **Convertir a `category` dtype**: LightGBM, XGBoost y CatBoost lo soportan nativamente sin one-hot. Si más adelante usamos un modelo de sklearn, se aplica one-hot al momento de entrenar.

Antes de transformar, exploramos los valores actuales para validar la decisión.

In [120]:
categorical_cols = ['Material', 'Technology', 'Post Production', 'Finishing', 'Heat Treatment', 'part_type']

for c in categorical_cols:
    n_unique = df_step1[c].nunique(dropna=True)
    n_missing = df_step1[c].isna().sum()
    print(f'\n--- {c}  (n_unique={n_unique}, n_missing={n_missing}) ---')
    print(df_step1[c].value_counts(dropna=False).head(15).to_string())

print('\n--- Tolerance (string que debería ser numérico) ---')
print(df_step1['Tolerance'].value_counts(dropna=False).head(15).to_string())


--- Material  (n_unique=22, n_missing=0) ---
Material
Other                   197
4140T Steel              39
D2 Steel                 34
6061 Aluminum            34
A2 Steel                 32
6061-T6 Aluminum         27
304 Stainless Steel      20
1018 Steel                5
D11 Sintered Steel        4
8620 Steel                4
PLA 80 Micrometers        3
7075-T6 Aluminum          2
17-4 Stainless Steel      2
416 Stainless Steel       2
7075 Aluminum             2

--- Technology  (n_unique=6, n_missing=0) ---
Technology
CNC Multiaxis       363
Other                21
3D Printing FDM      17
3D Printing SLS       6
3D Printing DMLS      5
Sheet Metal           5

--- Post Production  (n_unique=10, n_missing=249) ---
Post Production
NaN                                     249
Other                                   122
Laser Engraving                          20
Anodizing II                              7
Black Oxide Coating                       6
Laser Engraving, Black Oxide Coa

In [121]:
RARE_THRESHOLD = 5
RARE_LABEL = 'Other_rare'

# 1) Tolerance: convertir a numérico (valores no parseables → NaN)
n_before_na = df_step1['Tolerance'].isna().sum()
df_step1['Tolerance'] = pd.to_numeric(df_step1['Tolerance'], errors='coerce')
n_after_na = df_step1['Tolerance'].isna().sum()
print(f'Tolerance → float (NaN antes: {n_before_na}, después: {n_after_na})')

# 2) Agrupar categorías raras
print('\n--- Agrupando categorías raras (≤ {} ocurrencias) ---'.format(RARE_THRESHOLD))
for c in categorical_cols:
    counts = df_step1[c].value_counts(dropna=True)
    rare_values = counts[counts <= RARE_THRESHOLD].index.tolist()
    if rare_values:
        n_rows = df_step1[c].isin(rare_values).sum()
        df_step1.loc[df_step1[c].isin(rare_values), c] = RARE_LABEL
        print(f'  {c}: {len(rare_values)} categorías raras ({n_rows} filas) → {RARE_LABEL!r}')
    else:
        print(f'  {c}: ninguna categoría rara')

# 3) Convertir a category dtype (LightGBM/XGBoost/CatBoost lo aprovechan nativamente)
for c in categorical_cols:
    df_step1[c] = df_step1[c].astype('category')

print('\n--- dtypes y n_unique finales ---')
for c in categorical_cols:
    print(f'  {c}: dtype={df_step1[c].dtype}, n_unique={df_step1[c].nunique()}')

Tolerance → float (NaN antes: 0, después: 134)

--- Agrupando categorías raras (≤ 5 ocurrencias) ---
  Material: 15 categorías raras (34 filas) → 'Other_rare'
  Technology: 2 categorías raras (10 filas) → 'Other_rare'
  Post Production: 6 categorías raras (13 filas) → 'Other_rare'
  Finishing: 1 categorías raras (5 filas) → 'Other_rare'
  Heat Treatment: 2 categorías raras (2 filas) → 'Other_rare'
  part_type: 1 categorías raras (2 filas) → 'Other_rare'

--- dtypes y n_unique finales ---
  Material: dtype=category, n_unique=8
  Technology: dtype=category, n_unique=5
  Post Production: dtype=category, n_unique=5
  Finishing: dtype=category, n_unique=5
  Heat Treatment: dtype=category, n_unique=4
  part_type: dtype=category, n_unique=4


## 11. Eliminar duplicados

Tras todos los drops y filtros, varias filas pueden haber quedado idénticas — el preview inicial mostraba `RAD-6280` aparecer dos veces con typo (`Grippper` / `Gripper`); como ya no tenemos `Radii ID` ni `Line Item Name`, esos rows ahora son indistinguibles.

Con sólo ~400 filas, incluso pocos duplicados:
- Sesgan el train/test split (la misma fila puede caer en ambos lados → leakage en evaluación).
- Inflan artificialmente la confianza del modelo en patrones repetidos.

Detectamos duplicados sobre todas las columnas (incluyendo categóricas convertidas) y los eliminamos.

In [122]:
n_before = len(df_step1)
n_duplicates = df_step1.duplicated().sum()

df_step1 = df_step1.drop_duplicates().reset_index(drop=True)
n_after = len(df_step1)

print(f'Registros antes:        {n_before}')
print(f'Filas duplicadas:       {n_duplicates}')
print(f'Registros después:      {n_after}')
print(f'Duplicados eliminados:  {n_before - n_after}')

Registros antes:        417
Filas duplicadas:       31
Registros después:      386
Duplicados eliminados:  31


## 12. Feature engineering: `log(Quantity)`

`Quantity` va de 1 a varios miles (≥3 órdenes de magnitud). Los descuentos por volumen son típicamente **multiplicativos** (-10%, -25%, ...): en escala lineal los árboles del GBM necesitan muchos splits para capturarlos; en escala log la curva se vuelve cuasi-lineal y el modelo la aprende con menos profundidad — especialmente útil con sólo ~400 filas.

Mantenemos también `Quantity` original por si el modelo encuentra señal en escala lineal (efectos de saturación, mínimos de pedido, etc.).

In [123]:
df_step1['log_quantity'] = np.log(df_step1['Quantity'])

print('--- Quantity (lineal) ---')
print(df_step1['Quantity'].describe())
print()
print('--- log_quantity ---')
print(df_step1['log_quantity'].describe())

--- Quantity (lineal) ---
count       386.000000
mean       5872.360104
std       35365.652231
min           1.000000
25%           1.000000
50%           2.000000
75%          12.000000
max      376000.000000
Name: Quantity, dtype: float64

--- log_quantity ---
count    386.000000
mean       1.612085
std        2.354418
min        0.000000
25%        0.000000
50%        0.693147
75%        2.484907
max       12.837344
Name: log_quantity, dtype: float64


## 13. Inventario final de missingness

Resumen de los `NaN` que quedan en el dataset listo para entrenar. LightGBM, XGBoost y CatBoost manejan NaN nativamente — pero saber dónde están sirve para:
- Decidir si vale la pena imputar alguna columna específica (mediana, `'Missing'` para categóricas).
- Escoger modelo (un sklearn vanilla requeriría imputación previa).
- Detectar patrones (p.ej. si una fuente de datos cae junta).

In [124]:
nan_summary = pd.DataFrame({
    'dtype': df_step1.dtypes.astype(str),
    'n_missing': df_step1.isna().sum(),
    'pct_missing': (df_step1.isna().mean() * 100).round(1),
})
nan_summary = nan_summary[nan_summary['n_missing'] > 0].sort_values('pct_missing', ascending=False)

print(f'Shape final: {df_step1.shape}')
print(f'Columnas con NaN: {len(nan_summary)} de {df_step1.shape[1]}')
print()
print(nan_summary.to_string())

Shape final: (386, 24)
Columnas con NaN: 4 de 24

                    dtype  n_missing  pct_missing
Heat Treatment   category        293         75.9
Post Production  category        229         59.3
Tolerance         float64        124         32.1
part_type        category          5          1.3


## 14. Guardar intermedio

In [125]:
df_step1.to_parquet(OUT_PATH, index=False)
print(f'Guardado: {OUT_PATH}  ({df_step1.shape[0]} filas, {df_step1.shape[1]} columnas)')

Guardado: line_items_step1.parquet  (386 filas, 24 columnas)


In [126]:
CSV_PREVIEW_PATH = Path('line_items_step1_preview.csv')
df_step1.to_csv(CSV_PREVIEW_PATH, index=False)

print(f'CSV de inspección: {CSV_PREVIEW_PATH}')
print(f'  filas:    {df_step1.shape[0]}')
print(f'  columnas: {df_step1.shape[1]}')
print(f'\nColumnas restantes:')
for c in df_step1.columns:
    print(f'  - {c}')

CSV de inspección: line_items_step1_preview.csv
  filas:    386
  columnas: 24

Columnas restantes:
  - Material
  - Technology
  - Tolerance
  - Post Production
  - Finishing
  - Heat Treatment
  - Quantity
  - Unit Price (USD)
  - part_volume_cm3
  - stock_volume_cm3
  - material_removal_pct
  - stock_dimensions_mm.x
  - stock_dimensions_mm.y
  - stock_dimensions_mm.z
  - faces_qty
  - aspect_ratio
  - thin_ratio
  - plane_ratio
  - cyl_ratio
  - complex_ratio
  - total_holes
  - part_type
  - log_unit_price
  - log_quantity


## Próximo paso — empezar a entrenar

**Estado del dataset listo para modelar**:
- Target final: `log_unit_price` (con `Unit Price (USD)` preservado para inverse transform vía `np.exp`).
- Categóricas en `category` dtype, `Tolerance` numérico, `log_quantity` añadido.
- Sin duplicados, sin outliers extremos en el target.
- ~400 filas × 24 columnas (22 features + 2 targets).

**En el siguiente notebook (`02_modeling.ipynb`)**:

1. **Baseline triviales** para comparar:
   - Mediana del target en train (sin features).
   - Regresión sobre `log_quantity` sola (efecto de descuento por volumen aislado).
2. **Train/test split**: 80/20 estratificado por bins del target. Hacerlo ANTES de cualquier transformación derivada de los datos para evitar leakage.
3. **Modelo principal**: LightGBM con `categorical_feature='auto'` (detecta los `category` dtype). Métricas: MAE/RMSE en log-space, MAPE en USD tras inverse transform.
4. **Si MAPE > 25%**: probable que sea por la categoría sumidero `Material='Other'` (47% de filas). Considerar agregar `Tolerance` numérico binned, o features derivadas de geometría.
5. **Tuning**: si el baseline anda razonable, hacer CV (5-fold) + Optuna o GridSearch.

**Banderas de calidad de datos a tener presente** (no son fixes — son cosas a saber al interpretar resultados):
- `Material='Other'` en 47% de las filas (categoría sumidero del CRM): el modelo no puede distinguir materiales reales escondidos ahí.
- `Post Production` y `Heat Treatment` con 60–76% NaN: poca señal aportable.
- `'Other_rare'` (binización nuestra) coexiste con `'Other'` (valor original del CSV) en la misma columna: el modelo los trata como categorías distintas.